# Árboles binarios

En un entorno académico riguroso, la gestión de memoria dinámica es fundamental. A diferencia de las implementaciones heredadas de C que utilizan punteros crudos (*) y new/delete manual, emplearemos el paradigma RAII (Resource Acquisition Is Initialization) mediante std::unique_ptr para prevenir fugas de memoria (memory leaks) de forma determinista y segura.

In [1]:
#include <iostream>
#include <memory>
#include <utility>

// Definición genérica del Nodo empleando templates para type-safety
template <typename T>
struct BinaryTreeNode {
    T data;
    // std::unique_ptr garantiza la propiedad exclusiva de los subárboles.
    // Al destruirse el nodo, se destruyen en cascada sus hijos (O(n) tiempo, O(1) código).
    std::unique_ptr<BinaryTreeNode<T>> left;
    std::unique_ptr<BinaryTreeNode<T>> right;

    // Constructor explícito para prevenir conversiones de tipo implícitas.
    // Uso de std::move para optimizar la transferencia de recursos si T es un tipo pesado.
    explicit BinaryTreeNode(T value) 
        : data(std::move(value)), left(nullptr), right(nullptr) {}
};

void ejemplo_01() {
    // Instanciación de la raíz utilizando std::make_unique para mayor seguridad y eficiencia
    auto root = std::make_unique<BinaryTreeNode<int>>(10);
    
    // Construcción de la jerarquía básica
    root->left = std::make_unique<BinaryTreeNode<int>>(5);
    root->right = std::make_unique<BinaryTreeNode<int>>(15);

    std::cout << "Topología inicializada. Nodo raíz: " << root->data << "\n";
    std::cout << "Hijo izquierdo: " << root->left->data << "\n";
    std::cout << "Hijo derecho: " << root->right->data << "\n";

    // No se requiere 'delete'. La memoria se libera automáticamente al salir del scope.
}

ejemplo_01();

Topología inicializada. Nodo raíz: 10
Hijo izquierdo: 5
Hijo derecho: 15


1. ***Templates (`template <typename T>`)***: Proporciona polimorfismo paramétrico. El compilador genera la estructura específica de tipo sin penalización de rendimiento en tiempo de ejecución.
   
2. ***Gestión de Memoria y Complejidad Espacial:*** `std::unique_ptr` impone propiedad estricta. El tamaño en memoria de la estructura base será el tamaño del tipo T más dos punteros (típicamente 16 bytes en arquitecturas de 64 bits). La destrucción del árbol toma un tiempo $O(n)$ donde $n$ es el número de nodos, pero la invocación es implícita y recursiva por la semántica del destructor de unique_ptr.

3. ***Caché y Localidad espacial:*** Es imperativo recalcar que, al alojarse dinámicamente (`make_unique` utiliza el heap), los nodos no tienen contigüidad en memoria. Esto genera una menor localidad de caché (cache misses) en comparación con un `std::vector`, un trade-off necesario por la flexibilidad estructural.

# Jerarquías y dependencias 

Para motivar el estudio de los árboles binarios y, críticamente, de sus recorridos, planteamos un problema fundamental en la teoría de compiladores: la evaluación de expresiones aritméticas. Una expresión como `(3 + 4) * 5`no puede ser evaluada de forma puramente secuencial de izquierda a derecha sin la gestión explícita de la precedencia de operadores mediante pilas adicionales. Sin embargo, al modelar la expresión como un Árbol de Sintaxis Abstracta (AST), la precedencia queda codificada intrínsecamente en la estructura geométrica del árbol.

In [2]:
#include <iostream>
#include <memory>
#include <stdexcept>
#include <string>

// Estructura de nodo polimórfica simplificada para representar un AST
struct ExprNode {
    bool is_operator;
    char op;       // Operador (si is_operator == true)
    int value;     // Operando (si is_operator == false)
    
    std::unique_ptr<ExprNode> left;
    std::unique_ptr<ExprNode> right;

    // Constructor para nodos Operando (Hojas)
    explicit ExprNode(int val) 
        : is_operator(false), op('\0'), value(val), left(nullptr), right(nullptr) {}

    // Constructor para nodos Operador (Nodos Internos)
    explicit ExprNode(char op_char) 
        : is_operator(true), op(op_char), value(0), left(nullptr), right(nullptr) {}
};

// Función de evaluación basada en un recorrido estricto en Post-orden (LRN)
int evaluateExpression(const ExprNode* node) {
    if (!node) return 0;

    // Caso base: Si es un operando (hoja), retornamos su valor directamente.
    if (!node->is_operator) {
        return node->value;
    }

    // Paso Recursivo (Post-orden):
    // 1. Visitar/Evaluar subárbol Izquierdo (Left)
    int left_val = evaluateExpression(node->left.get());
    
    // 2. Visitar/Evaluar subárbol Derecho (Right)
    int right_val = evaluateExpression(node->right.get());

    // 3. Procesar el Nodo actual (Node) aplicando el operador a los resultados de sus hijos.
    switch (node->op) {
        case '+': return left_val + right_val;
        case '*': return left_val * right_val;
        case '-': return left_val - right_val;
        default: throw std::invalid_argument("Operador no soportado");
    }
}

void ejemplo_02() {
    // Modelando la expresión: (3 + 4) * 5
    // La raíz es el operador de menor precedencia evaluativa (el último en ejecutarse).
    auto root = std::make_unique<ExprNode>('*');
    
    root->left = std::make_unique<ExprNode>('+');
    root->left->left = std::make_unique<ExprNode>(3);
    root->left->right = std::make_unique<ExprNode>(4);
    
    root->right = std::make_unique<ExprNode>(5);

    int result = evaluateExpression(root.get());
    std::cout << "El resultado de (3 + 4) * 5 evaluado en post-orden es: " << result << "\n";
}

ejemplo_02();

El resultado de (3 + 4) * 5 evaluado en post-orden es: 35


# Interfaz y Estructura Robusta

Definiremos una interfaz `ITree<T>` que establece las operaciones esenciales. El uso de virtual y = 0 (virtual puro) obliga a las clases derivadas a implementar estos métodos. Es crucial incluir un destructor virtual en la interfaz; de lo contrario, al destruir un objeto a través de un puntero a la interfaz, no se llamaría al destructor de la clase derivada, rompiendo la cadena de destrucción de `std::unique_ptr` y causando fugas de memoria.

Implementación con Interfaz (C++20):
cpp
```
#include <iostream>
#include <memory>
#include <utility>

// 1. Definición de la Interfaz (Contrato)
template <typename T>
class ITree {
public:
    // Destructor virtual puro: Garantiza la limpieza correcta en clases derivadas
    virtual ~ITree() = default;

    // Operaciones esenciales que cualquier árbol debe cumplir
    virtual void insert(T value) = 0;
    virtual void clear() = 0;
    // Los recorridos se definirán como métodos virtuales aquí más adelante
};

// 2. Implementación Concreta: BinaryTree
template <typename T>
class BinaryTree : public ITree<T> {
private:
    struct Node {
        T data;
        std::unique_ptr<Node> left;
        std::unique_ptr<Node> right;

        explicit Node(T val) 
            : data(std::move(val)), left(nullptr), right(nullptr) {}
    };

    std::unique_ptr<Node> root;

    // Método de soporte privado para la recursión
    void insertRecursive(std::unique_ptr<Node>& current, T value) {
        if (!current) {
            current = std::make_unique<Node>(std::move(value));
            return;
        }
        if (value < current->data) 
            insertRecursive(current->left, std::move(value));
        else 
            insertRecursive(current->right, std::move(value));
    }

public:
    BinaryTree() : root(nullptr) {}

    // Implementación de los métodos de la interfaz (override)
    void insert(T value) override {
        insertRecursive(root, std::move(value));
    }

    void clear() override {
        root.reset(); // RAII: Al resetear la raíz, se dispara la destrucción en cascada
    }
};

int main() {
    // Uso polimórfico: Programamos hacia la interfaz, no hacia la implementación
    std::unique_ptr<ITree<int>> myTree = std::make_unique<BinaryTree<int>>();
    
    myTree->insert(50);
    myTree->insert(30);
    myTree->insert(70);

    std::cout << "Interfaz implementada y árbol instanciado con éxito.\n";
    // Al salir de main, myTree se destruye, llamando al destructor virtual y liberando todo.
    return 0;
}
```

1. ***Polimorfismo en Tiempo de Ejecución:*** El uso de `std::unique_ptr<ITree<int>>` permite que el código cliente sea agnóstico a la implementación. Podríamos cambiar `BinaryTree` por `AVLTree` sin modificar la lógica del `main`.

2. ***override Keyword:*** Es una buena práctica de C++11/17/20. Indica explícitamente al compilador que tenemos la intención de sobrescribir un método virtual. Si la firma en la interfaz cambia, el compilador generará un error, evitando errores silenciosos.

3. ***Encapsulación de la Estructura Node:*** El nodo sigue siendo un detalle privado. La interfaz solo habla de valores T, manteniendo la abstracción de datos pura.

4. ***Gestión de Memoria y RAII:*** La función `clear()` utiliza `root.reset()`. Esto es semánticamente equivalente a decir "el árbol ya no es dueño de nada". El destructor de `unique_ptr` se encarga del resto de forma determinista.

# Recorridos en profundidad 

La topología no lineal de un árbol impide un recorrido iterativo simple mediante índices, como ocurre en un `std::vector`. Para explorar cada nodo, empleamos algoritmos de Búsqueda en Profundidad (DFS - Depth-First Search). Estos algoritmos se basan en la naturaleza recursiva del árbol: un árbol está compuesto por una raíz y dos subárboles disjuntos.

Dependiendo del momento exacto en que procesamos (visitamos) el nodo raíz en relación con sus subárboles, emergen tres permutaciones canónicas:

* ***Pre-orden (NLR - Node, Left, Right):*** Visitar el nodo, recursión izquierda, recursión derecha. Útil para clonar topologías.

* ***In-orden (LNR - Left, Node, Right):*** Recursión izquierda, visitar el nodo, recursión derecha. En un Árbol Binario de Búsqueda (BST), este recorrido garantiza la extracción de los datos en orden monotónico ascendente.

* ***Post-orden (LRN - Left, Right, Node):*** Recursión izquierda, recursión derecha, visitar el nodo. Como vimos en la motivación, es estrictamente necesario cuando el procesamiento de un nodo depende del resultado de sus descendientes (e.g., destrucción de memoria, evaluación de AST).

In [3]:
#include <iostream>
#include <memory>
#include <functional>

// Extensión de la Interfaz ITree
template <typename T>
class ITree {
public:
    virtual ~ITree() = default;
    virtual void insert(T value) = 0;
    
    // El uso de std::function permite inyectar el comportamiento (Visitor Pattern funcional)
    // Se declaran como 'const' porque recorrer el árbol no debe alterar su estructura.
    virtual void preOrder(std::function<void(const T&)> action) const = 0;
    virtual void inOrder(std::function<void(const T&)> action) const = 0;
    virtual void postOrder(std::function<void(const T&)> action) const = 0;
};

// Extensión de BinaryTree
template <typename T>
class BinaryTree : public ITree<T> {
private:
    struct Node {
        T data;
        std::unique_ptr<Node> left;
        std::unique_ptr<Node> right;
        explicit Node(T val) : data(std::move(val)), left(nullptr), right(nullptr) {}
    };

    std::unique_ptr<Node> root;

    // Métodos recursivos de soporte (Privados)
    // Utilizamos raw pointers (const Node*) para la navegación sin afectar la propiedad
    void preOrderRecursive(const Node* current, const std::function<void(const T&)>& action) const {
        if (!current) return;
        action(current->data);                                 // N (Procesar)
        preOrderRecursive(current->left.get(), action);        // L (Izquierda)
        preOrderRecursive(current->right.get(), action);       // R (Derecha)
    }

    void inOrderRecursive(const Node* current, const std::function<void(const T&)>& action) const {
        if (!current) return;
        inOrderRecursive(current->left.get(), action);         // L (Izquierda)
        action(current->data);                                 // N (Procesar)
        inOrderRecursive(current->right.get(), action);        // R (Derecha)
    }

    void postOrderRecursive(const Node* current, const std::function<void(const T&)>& action) const {
        if (!current) return;
        postOrderRecursive(current->left.get(), action);       // L (Izquierda)
        postOrderRecursive(current->right.get(), action);      // R (Derecha)
        action(current->data);                                 // N (Procesar)
    }

    void insertRecursive(std::unique_ptr<Node>& current, T value) {
        if (!current) { current = std::make_unique<Node>(std::move(value)); return; }
        if (value < current->data) insertRecursive(current->left, std::move(value));
        else insertRecursive(current->right, std::move(value));
    }

public:
    BinaryTree() : root(nullptr) {}
    void insert(T value) override { insertRecursive(root, std::move(value)); }

    // Implementación pública de los recorridos
    void preOrder(std::function<void(const T&)> action) const override {
        preOrderRecursive(root.get(), action);
    }
    void inOrder(std::function<void(const T&)> action) const override {
        inOrderRecursive(root.get(), action);
    }
    void postOrder(std::function<void(const T&)> action) const override {
        postOrderRecursive(root.get(), action);
    }
};

void ejemplo_04() {
    BinaryTree<int> tree;
    // Inserción para construir un BST
    for (int val : {50, 30, 70, 20, 40, 60, 80}) tree.insert(val);

    // Definimos la acción mediante una expresión Lambda
    auto printAction = [](const int& val) { std::cout << val << " "; };

    std::cout << "In-orden (LNR - Imprime ordenado): ";
    tree.inOrder(printAction); 
    std::cout << "\n";
}

ejemplo_04();

In-orden (LNR - Imprime ordenado): 20 30 40 50 60 70 80 


# Evaluación de Casos Extremos y Límites Físicos



In [ ]:
#include <iostream>
#include <memory>
#include <functional>

// Estructura mínima de prueba basada en nuestra arquitectura previa
struct Node {
    int data;
    std::unique_ptr<Node> right; // Solo usaremos el hijo derecho para forzar degeneración
    explicit Node(int val) : data(val), right(nullptr) {}
};

// Función recursiva vulnerable al límite del Call Stack (O(h) espacio)
void vulnerableInOrder(const Node* node, int& counter) {
    if (!node) return;
    // Omitimos hijo izquierdo porque sabemos que es nulo en este test
    counter++; 
    vulnerableInOrder(node->right.get(), counter);
}

void ejemplo_05() {
    std::cout << "Iniciando prueba de estrés topológico...\n";
    
    auto root = std::make_unique<Node>(0);
    Node* current = root.get();
    
    // Inserción iterativa para no reventar la pila durante la construcción
    // Intentamos crear una profundidad h = 100,000 (Típico límite en Linux con 8MB de Stack)
    const int N_NODES = 100000; 
    for (int i = 1; i < N_NODES; ++i) {
        current->right = std::make_unique<Node>(i);
        current = current->right.get();
    }
    
    std::cout << "Árbol degenerado construido en el Heap. Altura h = " << N_NODES << ".\n";
    
    int nodes_visited = 0;
    try {
        // Al ejecutar esta línea en un entorno real sin optimización de recursión de cola (TCO),
        // el programa sufrirá un Segmentation Fault (Stack Overflow).
        vulnerableInOrder(root.get(), nodes_visited);
        std::cout << "Recorrido completado. Nodos visitados: " << nodes_visited << "\n";
    } catch (...) {
        // Nota técnica: Stack Overflow en C++ no lanza una excepción estándar atrapable (std::exception).
        // Resulta en un fallo a nivel del sistema operativo (SIGSEGV).
        std::cerr << "Fallo del sistema detectado durante la recursión.\n";
    }

    // RAII actúa aquí: Destrucción en cascada O(n). 
    // ¡CUIDADO! La destrucción en cascada de unique_ptr TAMBIÉN es recursiva en su destructor predeterminado.
    // Un árbol degenerado de este tamaño causará un Stack Overflow al intentar destruirse a sí mismo.
}

ejemplo_05();